# Depth Anything V3 (ByteDance)

Metric depth.

**Runtime:** GPU (L4 or A100)

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, random, time, gc, sys, json, importlib, subprocess
import numpy as np
import cv2
import torch
from PIL import Image

DATASET_IMAGES = '/content/drive/MyDrive/Corn Seed Dataset/test/images'
DATASET_LABELS = '/content/drive/MyDrive/Corn Seed Dataset/test/labels'
SAVE_DIR       = '/content/drive/MyDrive/Corn Seed Dataset/depth_comparison_outputs'
SAMPLE_FILE    = os.path.join(SAVE_DIR, 'sample_images.txt')

assert os.path.isdir(DATASET_IMAGES), f'Dataset not found: {DATASET_IMAGES}'
os.makedirs(SAVE_DIR, exist_ok=True)

def save_depth(depth_np, stem, model_name):
    d = np.array(depth_np, dtype=np.float32)
    while d.ndim > 2: d = d[0]
    d_norm = (d - d.min()) / (d.max() - d.min() + 1e-8)
    for sub, img in [('depth', (d_norm*65535).astype(np.uint16)),
                     ('vis',   cv2.applyColorMap((d_norm*255).astype(np.uint8), cv2.COLORMAP_INFERNO))]:
        p = os.path.join(SAVE_DIR, model_name, sub)
        os.makedirs(p, exist_ok=True)
        cv2.imwrite(os.path.join(p, f'{stem}.png'), img)

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('Setup done.')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Load shared sample list from Drive (run depth_pro.ipynb first)
assert os.path.exists(SAMPLE_FILE), f'Run depth_pro.ipynb first to create sample_images.txt'
SAMPLES = [l.strip() for l in open(SAMPLE_FILE) if l.strip()]
print(f'Loaded {len(SAMPLES)} samples')

## Install + run

In [ ]:
!pip install -q --force-reinstall 'utils3d @ git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec1'
!git clone https://github.com/ByteDance-Seed/Depth-Anything-3 /content/Depth-Anything-3 2>/dev/null || true
!pip install -q -e /content/Depth-Anything-3

importlib.invalidate_caches()
for k in [k for k in sys.modules if 'utils3d' in k or 'depth_anything' in k]:
    del sys.modules[k]

from depth_anything_3.api import DepthAnything3
MODEL_NAME = 'depth_anything_v3'
model = DepthAnything3.from_pretrained('depth-anything/da3metric-large').cuda().eval()

times = []
for img_name in SAMPLES:
    stem = img_name.split('.')[0]
    t0 = time.time()
    with torch.no_grad():
        pred = model.inference([os.path.join(DATASET_IMAGES, img_name)])
    torch.cuda.synchronize(); times.append(time.time()-t0)
    save_depth(pred.depth[0], stem, MODEL_NAME)

print(f'Done -- {len(times)} images, avg {np.mean(times):.3f}s/img')
del model; clear_gpu()

## Confirm saved

In [ ]:
model_dir = os.path.join(SAVE_DIR, MODEL_NAME)
n_depth = len(os.listdir(os.path.join(model_dir, 'depth')))
n_vis   = len(os.listdir(os.path.join(model_dir, 'vis')))
print(f'{MODEL_NAME}: {n_depth} depth maps, {n_vis} visualizations saved to Drive')